In [1]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖
import os
from matplotlib.font_manager import fontManager, FontProperties

fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name
from tqdm import tqdm

In [17]:
names = ['15_C03627F5_WT202403180043.h5ad',
         '17_C03627F6_WT202403270557.h5ad',
'19_D03657F1_WT202403110530.h5ad',
'21_D03657F2_WT202403110531.h5ad',
'22_B03606C4E6_WT202403310050.h5ad',
'23_B03609A4D6_WT202404150263.h5ad',
# '27_B03610C1E3_WT202403310051.h5ad',
# '31_B03619A1D3_WT202403310052.h5ad',
# '35_B03619E4G6_WT202403310053.h5ad',
# '39_A03589A1D4_WT202403310046.h5ad',
# '43_A03590E1G4_WT202403310064.h5ad',
# '47_A03593C1F3_WT202403310068.h5ad',
# '51_B03605C2E5_WT202406020126.h5ad',
# '55_B03613E3G6_WT202403310069.h5ad',
# '59_B03612E4G6_WT202403310059.h5ad',
# '63_B03606C1E3_WT202403310061.h5ad',
# '67_A03595A1D3_WT202403310062.h5ad',
# '71_A03595A4D6_WT202403310063.h5ad',
# '76_D03656A5_WT202403280404.h5ad',
# '81_D03657C6_WT202403110520.h5ad',
# '85_B03611D2_WT202403110546.h5ad',
# '90_A03592D3_WT202403110532.h5ad',
# '95_B03602D1_WT202403110535.h5ad',
# '100_B03609G1_WT202403280406.h5ad',
#          'A03590A3D6_WT202407192652.h5ad', # gw13
#          'A03588A1C2_WT202407161185.h5ad', # gw13
#          'A03988A1C2_WT202407161208.h5ad', # gw13
#          'A03994F1G2_WT2024071215067.h5ad',# gw13
#          # 'A03591D4E5_WT2024071215074.h5ad',
#          'A03587A5C6_WT2024071215080.h5ad', # gw10
#          'B03607C4E6_WT2024071214941.h5ad', # gw12
#          'B03618D3F6_WT202407152793.h5ad', # gw16
#          'B04122A3F6_WT202407282762.h5ad', # gw18
]

In [8]:
import pandas as pd

celltype = ['Cere_sc_2', 'Cere_sc_17', 'Cere_sc_18', 'Cere_sc_19', 'Cere_sc_20', 'Cere_sc_21']
slice_code = '19_D03657F1_WT202403110530'

csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/20251103/2_cerebellum_single/cci_20251112/{slice_code}.csv',
                  index_col='Unnamed: 0')

# 构造正则表达式模式，匹配任意一个 celltype
pattern = '|'.join(celltype)

# 使用 str.contains 进行筛选
csv = csv[csv['sr_pair'].str.contains(pattern, na=False)].copy()

In [18]:
celltype = ['Cere_sc_18', 'Cere_sc_2', 'Cere_sc_20', 'Cere_sc_21']
lr_pairs = []
print(celltype)
for name in names:
    try:
        slice_code = name.replace('.h5ad', '')
        csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/20251103/2_cerebellum_single/cci_20251112/{slice_code}.csv', index_col = 'Unnamed: 0')
        # csv = csv[csv['sr_pair'].apply(lambda x: celltype in x)]
        csv = csv[csv['sr_pair'].str.split('-').apply(lambda x: bool(set(x) & set(celltype)))]
        lr_pairs_temp = csv['lr_pair'].tolist()
        if not lr_pairs:
            lr_pairs = lr_pairs_temp
            print(1)
        else:
            lr_pairs = list(set(lr_pairs) & set(lr_pairs_temp))

    except Exception as e:
        print(e)
        continue

['Cere_sc_18', 'Cere_sc_2', 'Cere_sc_20', 'Cere_sc_21']
1


In [19]:
lr_pairs

['COL1A1-ITGB8',
 'CXCL12-CXCR4',
 'COL1A2-SDC4',
 'CNTN2-CNTN2',
 'NRG3-ERBB4',
 'COL1A2-ITGB8',
 'SEMA4C-PLXNB2',
 'COL1A1-SDC4',
 'CNTN2-CNTNAP2',
 'FN1-ITGA8']

In [21]:
lr_pair = ['COL1A1-ITGB8',
 'CXCL12-CXCR4',
 'COL1A2-SDC4',
 'CNTN2-CNTN2',
 'NRG3-ERBB4',
 'COL1A2-ITGB8',
 'SEMA4C-PLXNB2',
 'COL1A1-SDC4',
 'CNTN2-CNTNAP2',
 'FN1-ITGA8']

In [24]:
celltype = ['Cere_sc_2', 'Cere_sc_17', 'Cere_sc_18', 'Cere_sc_19', 'Cere_sc_20', 'Cere_sc_21']

In [25]:
names = [
    # '15_C03627F5_WT202403180043.h5ad',
         # '17_C03627F6_WT202403270557.h5ad',
'19_D03657F1_WT202403110530.h5ad',
# '21_D03657F2_WT202403110531.h5ad',
# '22_B03606C4E6_WT202403310050.h5ad',
# '23_B03609A4D6_WT202404150263.h5ad',
# '27_B03610C1E3_WT202403310051.h5ad',
# '31_B03619A1D3_WT202403310052.h5ad',
# '35_B03619E4G6_WT202403310053.h5ad',
# '39_A03589A1D4_WT202403310046.h5ad',
# '43_A03590E1G4_WT202403310064.h5ad',
# '47_A03593C1F3_WT202403310068.h5ad',
# '51_B03605C2E5_WT202406020126.h5ad',
# '55_B03613E3G6_WT202403310069.h5ad',
# '59_B03612E4G6_WT202403310059.h5ad',
# '63_B03606C1E3_WT202403310061.h5ad',
# '67_A03595A1D3_WT202403310062.h5ad',
# '71_A03595A4D6_WT202403310063.h5ad',
# '76_D03656A5_WT202403280404.h5ad',
# '81_D03657C6_WT202403110520.h5ad',
# '85_B03611D2_WT202403110546.h5ad',
# '90_A03592D3_WT202403110532.h5ad',
# '95_B03602D1_WT202403110535.h5ad',
# '100_B03609G1_WT202403280406.h5ad',
         'A03590A3D6_WT202407192652.h5ad', # gw13
         'A03588A1C2_WT202407161185.h5ad', # gw13
         'A03988A1C2_WT202407161208.h5ad', # gw13
         'A03994F1G2_WT2024071215067.h5ad',# gw13
#          # 'A03591D4E5_WT2024071215074.h5ad',
         'A03587A5C6_WT2024071215080.h5ad', # gw10
         'B03607C4E6_WT2024071214941.h5ad', # gw12
         'B03618D3F6_WT202407152793.h5ad', # gw16
         'B04122A3F6_WT202407282762.h5ad', # gw18
]

In [29]:

for name in names:
    try:
        slice_code = name.replace('.h5ad', '')
        csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/20251103/2_cerebellum_single/cci_20251112/{slice_code}.csv',
                          index_col='Unnamed: 0')

        csv = csv[csv['lr_pair'].isin(lr_pairs)]
        # csv = csv[True if 'Cere_sc_17' else False not in csv['sr_pair']]
        # csv = csv[~csv['sr_pair'].str.contains('Cere_sc_17', na=False)].copy()
        if len(csv) == 0:
            print(name)
            continue

        scaler = MinMaxScaler()
        csv['lr_co_exp_ratio_scaled'] = scaler.fit_transform(csv[['lr_co_exp_ratio']])

        # --- 关键：自定义 x 轴顺序 ---
        # 1. 先拿到所有出现的 sr_pair
        all_pairs = list(set(csv['sr_pair']))
        # 2. 定义优先级
        def priority(pair):
            # 只要包含指定群就返回优先级数字，越小越靠左
            if '2'  in pair.split('_'): return 0
            if '17' in pair.split('_'): return 1
            if '18' in pair.split('_'): return 2
            if '19' in pair.split('_'): return 3
            if '20' in pair.split('_'): return 4
            if '21' in pair.split('_'): return 5
            return 4   # 其余放最后
        # 3. 排序
        ordered_pairs = sorted(all_pairs, key=priority)
        # 4. 把 sr_pair 转成有序类别，保证 seaborn 按这个顺序画
        csv['sr_pair'] = pd.Categorical(csv['sr_pair'],
                                          categories=ordered_pairs,
                                          ordered=True)

        plt.figure(figsize=(len(ordered_pairs)/2+1,
                            len(set(csv['lr_pair']))/2+1))
        sns.scatterplot(x='sr_pair',
                        y='lr_pair',
                        size='lr_co_exp_ratio',
                        hue='lr_co_exp_ratio',
                        data=csv,
                        palette='viridis')
        handles, labels = plt.gca().get_legend_handles_labels()
        labels = [f"{float(label):.2f}" for label in labels]
        plt.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

        significant_points = csv[csv['is_significant'] == True]
        plt.scatter(x=significant_points['sr_pair'],
                    y=significant_points['lr_pair'],
                    facecolors='none', edgecolors='red', s=100)

        plt.xlim(-1, len(ordered_pairs)+1)
        plt.xticks(rotation=45, ha='right')
        plt.xlabel(None)
        plt.ylabel(None)
        plt.title('')
        plt.tight_layout()
        plt.savefig(f'/data/work/05.cluster/FuseMap/20251103/2_cerebellum_single/cci_plot_20251125/selected/{slice_code}.pdf',
                    bbox_inches='tight')
        plt.close()

    except Exception as e:
        print(e)

In [12]:
csv

""


In [28]:

for name in names:
    try:
        slice_code = name.replace('.h5ad', '')
        csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/20251103/2_cerebellum_single/cci_20251112/{slice_code}.csv',
                          index_col='Unnamed: 0')

        # csv = csv[csv['lr_pair'].isin(lr_pairs)]
        # csv = csv[True if 'Cere_sc_17' else False not in csv['sr_pair']]
        csv = csv[~csv['sr_pair'].str.contains('Cere_sc_17', na=False)].copy()
        if len(csv) == 0:
            print(name)
            continue

        scaler = MinMaxScaler()
        csv['lr_co_exp_ratio_scaled'] = scaler.fit_transform(csv[['lr_co_exp_ratio']])

        # --- 关键：自定义 x 轴顺序 ---
        # 1. 先拿到所有出现的 sr_pair
        all_pairs = list(set(csv['sr_pair']))
        # 2. 定义优先级
        def priority(pair):
            # 只要包含指定群就返回优先级数字，越小越靠左
            if '2'  in pair.split('_'): return 0
            if '17' in pair.split('_'): return 1
            if '18' in pair.split('_'): return 2
            if '19' in pair.split('_'): return 3
            if '20' in pair.split('_'): return 4
            if '21' in pair.split('_'): return 5
            return 4   # 其余放最后
        # 3. 排序
        ordered_pairs = sorted(all_pairs, key=priority)
        # 4. 把 sr_pair 转成有序类别，保证 seaborn 按这个顺序画
        csv['sr_pair'] = pd.Categorical(csv['sr_pair'],
                                          categories=ordered_pairs,
                                          ordered=True)

        plt.figure(figsize=(len(ordered_pairs)/2+1,
                            len(set(csv['lr_pair']))/2+1))
        sns.scatterplot(x='sr_pair',
                        y='lr_pair',
                        size='lr_co_exp_ratio',
                        hue='lr_co_exp_ratio',
                        data=csv,
                        palette='viridis')
        handles, labels = plt.gca().get_legend_handles_labels()
        labels = [f"{float(label):.2f}" for label in labels]
        plt.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

        significant_points = csv[csv['is_significant'] == True]
        plt.scatter(x=significant_points['sr_pair'],
                    y=significant_points['lr_pair'],
                    facecolors='none', edgecolors='red', s=100)

        plt.xlim(-1, len(ordered_pairs)+1)
        plt.xticks(rotation=45, ha='right')
        plt.xlabel(None)
        plt.ylabel(None)
        plt.title('')
        plt.tight_layout()
        plt.savefig(f'/data/work/05.cluster/FuseMap/20251103/2_cerebellum_single/cci_plot_20251125/all/{slice_code}.pdf',
                    bbox_inches='tight')
        plt.close()

    except Exception as e:
        print(e)